# Step 2: Label Harmonization

This notebook creates a standardized subject-and-recording metadata table for ADFTD, BrainLat, and AD-Auditory. It performs **label harmonization only**. It does not load EEG signal arrays, preprocess EEG, resample, filter, change channels, segment, combine signals, split subjects, or train a model.

Original labels are preserved exactly in `original_label`. The standardized labels are limited to `AD`, `FTD`, `HC`, `MCI`, and `UNKNOWN`. Unknown labels are never guessed or converted to healthy controls.

In [1]:
from pathlib import Path
import re

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 50)
pd.set_option("display.max_colwidth", 180)

NOTEBOOK_DIR = Path.cwd()
if NOTEBOOK_DIR.name != "NOTEBOOK":
    NOTEBOOK_DIR = Path(".").resolve()
PROJECT_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "NOTEBOOK" else NOTEBOOK_DIR

DATA_CANDIDATES = {
    "ADFTD": [PROJECT_ROOT / "data" / "ADFTD", PROJECT_ROOT / "SIH_Data" / "dt1"],
    "BrainLat": [PROJECT_ROOT / "data" / "BrainLat", PROJECT_ROOT / "SIH_Data" / "dt 2"],
    "AD_Auditory": [PROJECT_ROOT / "data" / "AD_Auditory", PROJECT_ROOT / "SIH_Data" / "dt 3"],
}

def first_existing(paths):
    return next((path for path in paths if path.exists()), paths[0])

DATA_ROOTS = {name: first_existing(paths) for name, paths in DATA_CANDIDATES.items()}
for dataset, root in DATA_ROOTS.items():
    print(f"{dataset}: {root} | exists={root.exists()}")

ADFTD: /home/admin1/Desktop/SIH_2026_ALL/SIH_Data/dt1 | exists=True
BrainLat: /home/admin1/Desktop/SIH_2026_ALL/SIH_Data/dt 2 | exists=True
AD_Auditory: /home/admin1/Desktop/SIH_2026_ALL/SIH_Data/dt 3 | exists=True


## 1. Load diagnosis metadata

The label mappings below are based on the actual downloaded metadata. ADFTD's `participants.tsv` defines `A` as Alzheimer disease, `F` as frontotemporal dementia, and `C` as healthy. BrainLat's demographic CSV files define `AD`, `FTD`, and `CN`. AD-Auditory's `participants.tsv` contains `Normal`, `Mild AD`, `MCI`, and unresolved values.

In [12]:
def read_table(path):
    separator = "\t" if path.suffix.lower() == ".tsv" else ","
    return pd.read_csv(path, sep=separator, dtype=str)

def load_participants(root):
    for filename in ("participants.tsv", "participants.csv"):
        path = root / filename
        if path.exists():
            return read_table(path), path
    return pd.DataFrame(), None

def standardize_label(dataset, value):
    if pd.isna(value):
        return "UNKNOWN"
    original = str(value).strip()
    if not original or original in {"-", "nan", "none"}:
        return "UNKNOWN"
    label = re.sub(r"[^a-z0-9]+", " ", original.lower()).strip()

    if dataset == "ADFTD":
        return {"a": "AD", "f": "FTD", "c": "HC"}.get(label, "UNKNOWN")
    if dataset == "BrainLat":
        return {"ad": "AD", "ftd": "FTD", "bvftd": "FTD", "cn": "HC", "hc": "HC"}.get(label, "UNKNOWN")

    if label == "mci" or "mild cognitive impairment" in label:
        return "MCI"
    if label in {"normal", "healthy", "healthy control", "control", "cn", "hc"}:
        return "HC"
    if label in {"ftd", "bvftd", "behavioral variant ftd", "behavioral variant frontotemporal dementia", "frontotemporal", "frontotemporal dementia"}:
        return "FTD"
    if label in {"ad", "alzheimer", "alzheimers", "alzheimer disease", "alzheimers disease"} or re.fullmatch(r"(mild|moderate|severe) ad", label):
        return "AD"
    return "UNKNOWN"

label_tables = {}
metadata_sources = {}
for dataset in ("ADFTD", "AD_Auditory"):
    participants, source = load_participants(DATA_ROOTS[dataset])
    metadata_sources[dataset] = source
    if participants.empty or not {"participant_id", "Group"}.issubset(participants.columns):
        label_tables[dataset] = pd.DataFrame(columns=["subject_id", "original_label", "standard_label"])
        continue
    labels = participants[["participant_id", "Group"]].copy()
    labels = labels.rename(columns={"participant_id": "subject_id", "Group": "original_label"})
    labels["standard_label"] = labels["original_label"].map(lambda value: standardize_label(dataset, value))
    label_tables[dataset] = labels

brainlat_root = DATA_ROOTS["BrainLat"]
brainlat_demographics = sorted(brainlat_root.rglob("Demographics*.csv"))
brainlat_parts = [read_table(path) for path in brainlat_demographics]
brainlat_metadata = pd.concat(brainlat_parts, ignore_index=True) if brainlat_parts else pd.DataFrame()
if {"id EEG", "diagnosis"}.issubset(brainlat_metadata.columns):
    labels = brainlat_metadata[["id EEG", "diagnosis"]].copy()
    labels = labels.rename(columns={"id EEG": "subject_id", "diagnosis": "original_label"})
    labels["standard_label"] = labels["original_label"].map(lambda value: standardize_label("BrainLat", value))
    label_tables["BrainLat"] = labels
else:
    label_tables["BrainLat"] = pd.DataFrame(columns=["subject_id", "original_label", "standard_label"])

for dataset, labels in label_tables.items():
    print(f"\n{dataset} metadata source: {metadata_sources.get(dataset, 'BrainLat demographic CSV files')}")
    display(labels[["original_label", "standard_label"]].value_counts(dropna=False).rename("subjects").reset_index())


ADFTD metadata source: /home/admin1/Desktop/SIH_2026_ALL/SIH_Data/dt1/participants.tsv


,original_label,standard_label,subjects
0,A,AD,36
1,C,HC,29
2,F,FTD,23



AD_Auditory metadata source: /home/admin1/Desktop/SIH_2026_ALL/SIH_Data/dt 3/participants.tsv


,original_label,standard_label,subjects
0,Mild AD,AD,16
1,Normal,HC,10
2,MCI,MCI,6
3,-,UNKNOWN,2
4,Moderate AD,AD,1



BrainLat metadata source: BrainLat demographic CSV files


,original_label,standard_label,subjects
0,CN,HC,46
1,AD,AD,35
2,FTD,FTD,19


## 2. Create one harmonized row per EEG recording

The table keeps every primary EEG file. EEGLAB `.fdt` files are treated as companion data files rather than separate recordings when their `.set` file is present. BrainLat recordings are restricted to the metadata-supported AD, FTD/bvFTD, and HC labels; PD and MS are excluded. Subject IDs are taken from the file naming structure only to match them to already loaded metadata, never to create a diagnosis.

In [13]:
PRIMARY_EXTENSIONS = {".edf", ".fif", ".bdf", ".set", ".vhdr", ".cnt", ".mff", ".con"}

def primary_recordings(root):
    return sorted([
        path for path in root.rglob("*")
        if path.is_file() and (path.suffix.lower() in PRIMARY_EXTENSIONS or path.name.lower().endswith(".fif.gz"))
    ])

def subject_from_filename(path):
    match = re.search(r"(sub[-_]?[A-Za-z0-9]+)", path.stem, flags=re.IGNORECASE)
    return match.group(1).replace("_", "-") if match else path.parent.name

harmonized_rows = []
included_brainlat_labels = {"AD", "FTD", "HC"}
for dataset, root in DATA_ROOTS.items():
    labels = label_tables[dataset].copy()
    if not labels.empty:
        labels["subject_key"] = labels["subject_id"].astype(str).str.replace("_", "-", regex=False).str.lower()
    label_lookup = labels.drop_duplicates("subject_key").set_index("subject_key").to_dict("index") if not labels.empty else {}
    for eeg_path in primary_recordings(root):
        subject_id = subject_from_filename(eeg_path)
        metadata = label_lookup.get(subject_id.lower(), {})
        original_label = metadata.get("original_label", np.nan)
        standard_label = metadata.get("standard_label", "UNKNOWN")
        if dataset == "BrainLat" and standard_label not in included_brainlat_labels:
            continue
        harmonized_rows.append({
            "dataset": dataset,
            "subject_id": subject_id,
            "original_label": original_label,
            "standard_label": standard_label,
            "eeg_file": str(eeg_path),
            "file_format": ".fif.gz" if eeg_path.name.lower().endswith(".fif.gz") else eeg_path.suffix.lower(),
        })

harmonized_metadata = pd.DataFrame(harmonized_rows, columns=["dataset", "subject_id", "original_label", "standard_label", "eeg_file", "file_format"])
harmonized_metadata = harmonized_metadata.sort_values(["dataset", "subject_id", "eeg_file"]).reset_index(drop=True)
print(f"Harmonized EEG recording rows: {len(harmonized_metadata)}")
display(harmonized_metadata.head(10))

Harmonized EEG recording rows: 310


,dataset,subject_id,original_label,standard_label,eeg_file,file_format
0,ADFTD,sub-001,A,AD,/home/admin1/Desktop/SIH_2026_ALL/SIH_Data/dt1/derivatives/sub-001/eeg/sub-001_task-eyesclosed_eeg.set,.set
1,ADFTD,sub-001,A,AD,/home/admin1/Desktop/SIH_2026_ALL/SIH_Data/dt1/sub-001/eeg/sub-001_task-eyesclosed_eeg.set,.set
2,ADFTD,sub-002,A,AD,/home/admin1/Desktop/SIH_2026_ALL/SIH_Data/dt1/derivatives/sub-002/eeg/sub-002_task-eyesclosed_eeg.set,.set
3,ADFTD,sub-002,A,AD,/home/admin1/Desktop/SIH_2026_ALL/SIH_Data/dt1/sub-002/eeg/sub-002_task-eyesclosed_eeg.set,.set
4,ADFTD,sub-003,A,AD,/home/admin1/Desktop/SIH_2026_ALL/SIH_Data/dt1/derivatives/sub-003/eeg/sub-003_task-eyesclosed_eeg.set,.set
5,ADFTD,sub-003,A,AD,/home/admin1/Desktop/SIH_2026_ALL/SIH_Data/dt1/sub-003/eeg/sub-003_task-eyesclosed_eeg.set,.set
6,ADFTD,sub-004,A,AD,/home/admin1/Desktop/SIH_2026_ALL/SIH_Data/dt1/derivatives/sub-004/eeg/sub-004_task-eyesclosed_eeg.set,.set
7,ADFTD,sub-004,A,AD,/home/admin1/Desktop/SIH_2026_ALL/SIH_Data/dt1/sub-004/eeg/sub-004_task-eyesclosed_eeg.set,.set
8,ADFTD,sub-005,A,AD,/home/admin1/Desktop/SIH_2026_ALL/SIH_Data/dt1/derivatives/sub-005/eeg/sub-005_task-eyesclosed_eeg.set,.set
9,ADFTD,sub-005,A,AD,/home/admin1/Desktop/SIH_2026_ALL/SIH_Data/dt1/sub-005/eeg/sub-005_task-eyesclosed_eeg.set,.set


## 3. Validate subject identity and labels

These checks are designed for later subject-wise splitting. Multiple recordings for one subject are allowed, but a subject must not receive conflicting standardized diagnoses within a dataset. No random IDs are created.

In [14]:
dataset_subject_counts = (
    harmonized_metadata.groupby("dataset")["subject_id"]
    .nunique()
    .rename("unique_subjects")
    .reset_index()
)
label_subject_counts = (
    harmonized_metadata.groupby("standard_label")["subject_id"]
    .nunique()
    .reindex(["AD", "FTD", "HC", "MCI", "UNKNOWN"], fill_value=0)
    .rename("total_subjects")
    .reset_index()
)

recordings_per_subject = (
    harmonized_metadata.groupby(["dataset", "subject_id"]).size()
    .rename("number_of_eeg_files")
    .reset_index()
)
duplicate_subjects = recordings_per_subject[recordings_per_subject["number_of_eeg_files"] > 1].copy()
conflicting_labels = (
    harmonized_metadata.groupby(["dataset", "subject_id"])["standard_label"]
    .nunique()
    .rename("number_of_standard_labels")
    .reset_index()
)
conflicting_labels = conflicting_labels[conflicting_labels["number_of_standard_labels"] > 1]
missing_diagnosis = harmonized_metadata[
    harmonized_metadata["original_label"].isna() | (harmonized_metadata["standard_label"] == "UNKNOWN")
].copy()
missing_paths = harmonized_metadata[
    harmonized_metadata["eeg_file"].isna() | (harmonized_metadata["eeg_file"].astype(str).str.strip() == "")
].copy()

print("Unique subjects by dataset:")
display(dataset_subject_counts)
print("Unique subjects by standardized diagnosis:")
display(label_subject_counts)
print(f"Subjects with multiple EEG files: {len(duplicate_subjects)}")
display(duplicate_subjects.head(20))
print(f"Subjects with conflicting standardized labels: {len(conflicting_labels)}")
display(conflicting_labels)
print(f"Rows with missing or UNKNOWN diagnosis: {len(missing_diagnosis)}")
display(missing_diagnosis)
print(f"Rows with missing EEG file paths: {len(missing_paths)}")
display(missing_paths)

brainlat_labels_present = set(harmonized_metadata.loc[harmonized_metadata["dataset"] == "BrainLat", "standard_label"].dropna())
brainlat_valid = brainlat_labels_present.issubset(included_brainlat_labels)
mci_subjects = set(harmonized_metadata.loc[harmonized_metadata["standard_label"] == "MCI", "subject_id"])
mci_mapped_to_ad = set(harmonized_metadata.loc[(harmonized_metadata["standard_label"] == "AD") & harmonized_metadata["original_label"].astype(str).str.contains("mci", case=False, na=False), "subject_id"])
print("BrainLat allowed-label check:", brainlat_valid, sorted(brainlat_labels_present))
print("MCI subjects preserved separately:", len(mci_subjects))
print("MCI labels mapped to AD:", sorted(mci_mapped_to_ad))
assert brainlat_valid
assert len(conflicting_labels) == 0
assert len(missing_paths) == 0
assert len(mci_mapped_to_ad) == 0

Unique subjects by dataset:


,dataset,unique_subjects
0,ADFTD,88
1,AD_Auditory,35
2,BrainLat,99


Unique subjects by standardized diagnosis:


,standard_label,total_subjects
0,AD,88
1,FTD,41
2,HC,85
3,MCI,6
4,UNKNOWN,2


Subjects with multiple EEG files: 88


...

Rows with missing or UNKNOWN diagnosis: 2


,dataset,subject_id,original_label,standard_label,eeg_file,file_format
181,AD_Auditory,sub-06,-,UNKNOWN,/home/admin1/Desktop/SIH_2026_ALL/SIH_Data/dt 3/sub-06/eeg/sub-06_task-40HzAuditoryEntrainment_eeg.set,.set
188,AD_Auditory,sub-13,-,UNKNOWN,/home/admin1/Desktop/SIH_2026_ALL/SIH_Data/dt 3/sub-13/eeg/sub-13_task-40HzAuditoryEntrainment_eeg.set,.set


Rows with missing EEG file paths: 0


,dataset,subject_id,original_label,standard_label,eeg_file,file_format


BrainLat allowed-label check: True ['AD', 'FTD', 'HC']
MCI subjects preserved separately: 6
MCI labels mapped to AD: []


## 4. Display summaries and save the harmonized metadata

The first summary counts subjects and EEG files by dataset and standardized label. The second summary keeps all five standard labels visible, including labels with zero subjects. The CSV is a new derived metadata file; no raw data is changed.

In [15]:
dataset_label_summary = (
    harmonized_metadata.groupby(["dataset", "standard_label"])
    .agg(number_of_subjects=("subject_id", "nunique"), number_of_eeg_files=("eeg_file", "count"))
    .reset_index()
)
standard_label_summary = label_subject_counts.rename(columns={"standard_label": "Standard Label", "total_subjects": "Total Subjects"})

print("SUMMARY: DATASET / STANDARD LABEL")
display(dataset_label_summary)
print("SUMMARY: STANDARD LABEL / TOTAL SUBJECTS")
display(standard_label_summary)

output_dir = PROJECT_ROOT / "processed_data"
output_dir.mkdir(parents=True, exist_ok=True)
output_path = output_dir / "harmonized_metadata.csv"
harmonized_metadata.to_csv(output_path, index=False)
print(f"Saved harmonized metadata: {output_path}")
saved_subjects = harmonized_metadata["subject_id"].nunique()
print(f"Saved rows: {len(harmonized_metadata)} | Saved subjects: {saved_subjects}")
print("LABEL HARMONIZATION COMPLETE — READY FOR EEG CHANNEL HARMONIZATION")

SUMMARY: DATASET / STANDARD LABEL


,dataset,standard_label,number_of_subjects,number_of_eeg_files
0,ADFTD,AD,36,72
1,ADFTD,FTD,23,46
2,ADFTD,HC,29,58
3,AD_Auditory,AD,17,17
4,AD_Auditory,HC,10,10
5,AD_Auditory,MCI,6,6
6,AD_Auditory,UNKNOWN,2,2
7,BrainLat,AD,35,35
8,BrainLat,FTD,18,18
9,BrainLat,HC,46,46


SUMMARY: STANDARD LABEL / TOTAL SUBJECTS


,Standard Label,Total Subjects
0,AD,88
1,FTD,41
2,HC,85
3,MCI,6
4,UNKNOWN,2


Saved harmonized metadata: /home/admin1/Desktop/SIH_2026_ALL/processed_data/harmonized_metadata.csv
Saved rows: 310 | Saved subjects: 222
LABEL HARMONIZATION COMPLETE — READY FOR EEG CHANNEL HARMONIZATION
